set project root + imports

In [1]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import mean_squared_error, r2_score

from xgboost import XGBRegressor

# --- ensure CWD is project root ---
# If running from notebooks/, go up one.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

# If still not at project root, walk until we find the data/ folder
p = Path.cwd()
while p != p.parent and not (p / "data").exists():
    p = p.parent
os.chdir(p)

print("CWD:", Path.cwd())

PROCESSED = Path("data/processed")
REPORTS = Path("reports")
FIGS = REPORTS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

CWD: c:\Users\josha\Codeacademy Projects\Data Scientist Portfolio Project


load processed expression + PRISM

In [2]:
expr_path = PROCESSED / "expression_tpm_logp1.parquet"
prism_path = PROCESSED / "prism_secondary_collapsed.parquet"

assert expr_path.exists(), f"Missing: {expr_path}"
assert prism_path.exists(), f"Missing: {prism_path}"

expr = pd.read_parquet(expr_path)
pr = pd.read_parquet(prism_path)

print("Expression:", expr.shape)
print("PRISM:", pr.shape)
pr.head()

Expression: (1517, 19194)
PRISM: (644798, 8)


,depmap_id,broad_id,auc,ic50,r2,ec50,slope,compound_name
0,ACH-000007,BRD-A00758722-001-04-9,0.903644,3.594623,0.272739,3.059164,3.457503,noretynodrel
1,ACH-000007,BRD-A02180903-001-04-5,0.938537,NaN,0.021503,0.125394,1.863805,betamethasone
2,ACH-000007,BRD-A03216249-003-24-3,1.457191,NaN,0.050635,0.000446,-0.226460,mepivacaine
3,ACH-000007,BRD-A03506276-001-01-5,0.512050,0.081393,0.977636,0.080971,6.584783,XL888
4,ACH-000007,BRD-A03623303-045-09-5,1.541743,NaN,-0.012929,0.000176,0.018727,metoprolol


sklearn-compatible TopVarianceSelector

In [3]:
class TopVarianceSelector(BaseEstimator, TransformerMixin):
    def __init__(self, top_n=2000):
        self.top_n = int(top_n)

    def fit(self, X, y=None):
        X_arr = X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)
        v = np.var(X_arr, axis=0)
        n = min(self.top_n, X_arr.shape[1])
        self.keep_idx_ = np.argsort(v)[::-1][:n]

        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        else:
            self.feature_names_in_ = None
        return self

    def transform(self, X):
        check_is_fitted(self, "keep_idx_")
        X_arr = X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)
        return X_arr[:, self.keep_idx_]

helpers: split + train + metrics

In [4]:
def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)

def grouped_train_val_test_split(X, y, groups, test_size=0.15, val_size=0.15, random_state=42):
    """
    Group-aware split: first split off test, then split remaining into train/val.
    val_size is fraction of full dataset (approximately).
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train = groups[train_idx]

    # Make val fraction of remaining set so overall val ~ val_size
    # remaining fraction = 1 - test_size
    # val_fraction_of_remaining = val_size / (1 - test_size)
    val_frac_remaining = min(0.5, val_size / (1 - test_size))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_frac_remaining, random_state=random_state)
    tr_idx, val_idx = next(gss2.split(X_train, y_train, groups=groups_train))

    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    return X_tr, X_val, X_test, y_tr, y_val, y_test


def fit_xgb_for_drug(df_model, target_col="auc", top_n_features=2000, random_state=42):
    """
    df_model must include columns: depmap_id, target_col, and many gene features.
    Returns dict of metrics + metadata.
    """
    y = df_model[target_col].astype(float)
    X = df_model.drop(columns=[target_col])

    groups = X["depmap_id"].astype(str).values
    X = X.drop(columns=["depmap_id"])

    X_tr, X_val, X_test, y_tr, y_val, y_test = grouped_train_val_test_split(
        X, y, groups, test_size=0.15, val_size=0.15, random_state=random_state
    )

    prep = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("selector", TopVarianceSelector(top_n=top_n_features)),
    ])

    X_tr_p = prep.fit_transform(X_tr)
    X_val_p = prep.transform(X_val)
    X_test_p = prep.transform(X_test)

    # XGBoost 3.x: early_stopping_rounds is set in constructor
    xgb = XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=random_state,
        n_jobs=-1,
        early_stopping_rounds=50,
    )

    xgb.fit(X_tr_p, y_tr.values, eval_set=[(X_val_p, y_val.values)], verbose=False)

    pred_val = xgb.predict(X_val_p)
    pred_test = xgb.predict(X_test_p)

    out = {
        "rmse_val": rmse(y_val, pred_val),
        "r2_val": float(r2_score(y_val, pred_val)),
        "rmse_test": rmse(y_test, pred_test),
        "r2_test": float(r2_score(y_test, pred_test)),
        "best_iteration": int(getattr(xgb, "best_iteration", -1)) if getattr(xgb, "best_iteration", None) is not None else None,
        "n_train": int(len(X_tr)),
        "n_val": int(len(X_val)),
        "n_test": int(len(X_test)),
        "n_features_in": int(X.shape[1]),
        "top_n_features": int(top_n_features),
    }
    return out

pick Top 10 drugs by coverage (n cell lines)

In [5]:
target_col = "auc"
assert target_col in pr.columns, "PRISM table missing 'auc' column."

coverage = (
    pr.groupby(["broad_id", "compound_name"])["depmap_id"]
      .nunique()
      .reset_index(name="n_cell_lines")
      .sort_values("n_cell_lines", ascending=False)
)

top10 = coverage.head(10).reset_index(drop=True)
top10

,broad_id,compound_name,n_cell_lines
0,BRD-K13662825-001-07-5,dinaciclib,480
1,BRD-K32744045-001-31-2,disulfiram,480
2,BRD-K95142244-001-01-5,talazoparib,480
3,BRD-K38852836-001-02-1,ganetespib,479
4,BRD-A25608658-001-01-1,narasin,479
5,BRD-K21361524-001-01-1,selinexor,479
6,BRD-K89714990-001-01-5,tepoxalin,479
7,BRD-K33622447-066-01-9,abemaciclib,479
8,BRD-K33379087-001-07-5,tivantinib,479
9,BRD-K31928526-001-02-1,barasertib,479


build modeling table per drug + train XGBoost + collect leaderboard

In [6]:
min_cell_lines = 200          # skip tiny drugs
top_n_features = 2000         # speed knob (try 1000 if runtime is slow)

rows = []

for i, row in top10.iterrows():
    broad_id = row["broad_id"]
    name = row["compound_name"]
    n = int(row["n_cell_lines"])

    if n < min_cell_lines:
        continue

    # response for this drug
    pr_drug = pr.loc[pr["broad_id"] == broad_id, ["depmap_id", target_col]].dropna()

    # merge with expression
    df_model = pr_drug.merge(expr, on="depmap_id", how="inner")

    # if merge shrinks too much, skip
    if len(df_model) < min_cell_lines:
        continue

    metrics = fit_xgb_for_drug(df_model, target_col=target_col, top_n_features=top_n_features, random_state=42)

    rows.append({
        "rank_by_coverage": i + 1,
        "compound_name": name,
        "broad_id": broad_id,
        "n_cell_lines_prism": n,
        "n_rows_modeling": int(len(df_model)),
        **metrics
    })

leaderboard = pd.DataFrame(rows).sort_values("rmse_test").reset_index(drop=True)
leaderboard[["rmse_val","rmse_test","r2_val","r2_test"]] = leaderboard[["rmse_val","rmse_test","r2_val","r2_test"]].round(4)
leaderboard

,rank_by_coverage,compound_name,broad_id,n_cell_lines_prism,n_rows_modeling,rmse_val,r2_val,rmse_test,r2_test,best_iteration,n_train,n_val,n_test,n_features_in,top_n_features
0,6,selinexor,BRD-K21361524-001-01-1,479,475,0.0737,0.0139,0.0561,0.0576,7,331,72,72,19193,2000
1,9,tivantinib,BRD-K33379087-001-07-5,479,475,0.0718,0.0586,0.0749,0.0275,63,331,72,72,19193,2000
2,1,dinaciclib,BRD-K13662825-001-07-5,480,476,0.0964,0.0654,0.0891,0.0810,25,332,72,72,19193,2000
3,4,ganetespib,BRD-K38852836-001-02-1,479,475,0.0988,0.0704,0.0982,0.0483,34,331,72,72,19193,2000
4,10,barasertib,BRD-K31928526-001-02-1,479,475,0.1073,0.0393,0.1001,-0.0167,73,331,72,72,19193,2000
5,7,tepoxalin,BRD-K89714990-001-01-5,479,475,0.1130,-0.0169,0.1109,0.0040,0,331,72,72,19193,2000
6,5,narasin,BRD-A25608658-001-01-1,479,475,0.1179,-0.0122,0.1126,-0.0534,1,331,72,72,19193,2000
7,8,abemaciclib,BRD-K33622447-066-01-9,479,475,0.0905,0.0272,0.1291,-0.0279,6,331,72,72,19193,2000
8,2,disulfiram,BRD-K32744045-001-31-2,480,476,0.1291,0.1198,0.1330,0.0657,106,332,72,72,19193,2000
9,3,talazoparib,BRD-K95142244-001-01-5,480,476,0.1494,0.2123,0.1535,0.2576,56,332,72,72,19193,2000


show mean test RMSE/R² across the top 10

In [7]:
mean_rmse = float(leaderboard["rmse_test"].mean())
std_rmse = float(leaderboard["rmse_test"].std(ddof=1)) if len(leaderboard) > 1 else 0.0
mean_r2 = float(leaderboard["r2_test"].mean())
std_r2 = float(leaderboard["r2_test"].std(ddof=1)) if len(leaderboard) > 1 else 0.0

print(f"Top-10 mean TEST RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}")
print(f"Top-10 mean TEST R²:   {mean_r2:.4f} ± {std_r2:.4f}")

Top-10 mean TEST RMSE: 0.1058 ± 0.0287
Top-10 mean TEST R²:   0.0444 ± 0.0868


save leaderboard to reports/ (CSV + JSON)

In [8]:
out_csv = REPORTS / "leaderboard_top10_auc_xgb.csv"
out_json = REPORTS / "leaderboard_top10_auc_xgb.json"

leaderboard.to_csv(out_csv, index=False)
with open(out_json, "w") as f:
    json.dump({
        "target": target_col,
        "min_cell_lines": min_cell_lines,
        "top_n_features": top_n_features,
        "n_drugs": int(len(leaderboard)),
        "mean_test_rmse": mean_rmse,
        "mean_test_r2": mean_r2,
        "leaderboard": leaderboard.to_dict(orient="records"),
    }, f, indent=2)

print("Wrote:", out_csv)
print("Wrote:", out_json)

Wrote: reports\leaderboard_top10_auc_xgb.csv
Wrote: reports\leaderboard_top10_auc_xgb.json
